### Overview ###

1. How CSVs are structured and how to read and write without the csv library.
2. Using the csv library to simplify CSV operations.
3. Using csv reader and DictReader to load and parse a dataset.

##### Step 1 #####

A tabular dataset of 100 meter times across 3 track meets can be represented as a nested list in Python.

In [1]:
track_times = [
    [13.10, 13.59, 13.44],
    [13.93, 13.85, 13.47],
    [14.12, 14.41, 13.89],
    [14.42, 13.55, 13.43]
]
track_times

[[13.1, 13.59, 13.44],
 [13.93, 13.85, 13.47],
 [14.12, 14.41, 13.89],
 [14.42, 13.55, 13.43]]

##### Step 2 #####

To convert this nested list to CSV:

1. Tabular data is converted to a string
2. Each row is given its own line of the string (separated by \n)
3. A comma separates each element in a row.

In [2]:
# initializing an empty string

track_times_csv = ""

In [4]:
# loop over each nested list

for index, athlete_times in enumerate(track_times):
    athlete_times_string = ",".join(str(time) for time in athlete_times) # join values in nested list separated by a comma
    track_times_csv += athlete_times_string # appends values to the empty string
    if index < (len(track_times) - 1):
        track_times += "\n"  # append a new line except on last row

track_times_csv

KeyboardInterrupt: 

##### Step 3 #####

Next, we write this string to a text file.

In [ ]:
with open("track_times.csv", "w") as f:
    f.write(track_times_csv)

##### Step 4 #####

Three operations are needed to parse the csv file (and the string it contains) into a nested list:

1. Split on newline characters, which is what readline() does
2. Split each line on the delimiter ","
3. Convert values to floats

In [ ]:
track_times_from_disk = []

with open("track_times.csv") as f:
    for row in f: # iterate over a list of lines (i.e. rows)
        times = [float(time) for time in row.split(",")] # list comprehension splits row into list, iterates over all elements to convert to float

        track_times_from_disk.append(times) # appends to row

track_times_from_disk

##### Step 5 #####

If everything above was done correctly, the new list should be identical to the original.

In [ ]:
track_times_from_disk == track_times

##### Step 6 #####

Processing real data can be quite a bit more complex:

- checking data types and conversions for a large number of columns
- processing headers
- properly handling data inside a csv (preserving e.g. quotes in text)
- representing data in different data structures (e.g. lists of dicts)

The csv module simplifies these tasks with reader functions and associated arguments.

In [5]:
import csv

**csv.reader**

In [ ]:
with open("track_times.csv") as f: # pass objects into a reader object that can be an iterator

    reader = csv.reader(f, quoting=csv.QUOTE_NONNUMERIC) # values w/o quotes treated as numbers

    track_times_with_csv_module = list(reader) # each element of the iterator has a fully processed line as a list

track_times_with_csv_module

The reader operates at a higher level of abstraction and does not require explicit string-cleaning operations. csv.reader has additional formatting capabilities for text files with different delimiters.

The csv module also has a parallel csv.writer object that writes lists to a csv file.

**csv.DictReader**

A challenge of nested lists is that column names are only in the first element of the outer list. Ideally the keys would be alongside the values in each row--i.e. each row as a dictionary.

csv.DictReader takes care of all of this without explicitly having to parse stringsd, create dictionaries, or perform list append operations.

##### Step 7 #####

Let's examine a real dataset of Olympic track-and-field winners. 

In [7]:
with open("results.csv") as f:
    reader = csv.reader(f)
    for _ in range(6):
        print(next(reader)) # printing header and first 5 rows

['Gender', 'Event', 'Location', 'Year', 'Medal', 'Name', 'Nationality', 'Result']
['M', '10000M Men', 'Rio', '2016', 'G', 'Mohamed FARAH', 'USA', '25:05.17']
['M', '10000M Men', 'Rio', '2016', 'S', 'Paul Kipngetich TANUI', 'KEN', '27:05.64']
['M', '10000M Men', 'Rio', '2016', 'B', 'Tamirat TOLA', 'ETH', '27:06.26']
['M', '10000M Men', 'Beijing', '2008', 'G', 'Kenenisa BEKELE', 'ETH', '27:01.17']
['M', '10000M Men', 'Beijing', '2008', 'S', 'Sileshi SIHINE', 'ETH', '27:02.77']


##### Step 8 #####

csv.DictReader parses rows into dicts, allowing easy lookups on column headers.

In [8]:
with open("results.csv") as f:
    reader = csv.DictReader(f) # creates an iterable
    olympics_data = list(reader) # converts to list

for index in range (5):
    print(olympics_data[index])

{'Gender': 'M', 'Event': '10000M Men', 'Location': 'Rio', 'Year': '2016', 'Medal': 'G', 'Name': 'Mohamed FARAH', 'Nationality': 'USA', 'Result': '25:05.17'}
{'Gender': 'M', 'Event': '10000M Men', 'Location': 'Rio', 'Year': '2016', 'Medal': 'S', 'Name': 'Paul Kipngetich TANUI', 'Nationality': 'KEN', 'Result': '27:05.64'}
{'Gender': 'M', 'Event': '10000M Men', 'Location': 'Rio', 'Year': '2016', 'Medal': 'B', 'Name': 'Tamirat TOLA', 'Nationality': 'ETH', 'Result': '27:06.26'}
{'Gender': 'M', 'Event': '10000M Men', 'Location': 'Beijing', 'Year': '2008', 'Medal': 'G', 'Name': 'Kenenisa BEKELE', 'Nationality': 'ETH', 'Result': '27:01.17'}
{'Gender': 'M', 'Event': '10000M Men', 'Location': 'Beijing', 'Year': '2008', 'Medal': 'S', 'Name': 'Sileshi SIHINE', 'Nationality': 'ETH', 'Result': '27:02.77'}


The number of rows in the dataset is just the length of the resulting list.

In [9]:
len(olympics_data)

2394

##### Step 9 #####

Data analysis and cleaning can now be achieved much more netaly and clearly.

Filtering data to only include gold medals:

In [10]:
gold_medals = []

for row in olympics_data:
    if row["Medal"] == "G":
        gold_medals.append(row)

print(f"""Out of {len(olympics_data)} total medals, this dataset contains {len(gold_medals)} gold medals""")

Out of 2394 total medals, this dataset contains 799 gold medals


To print USA gold medals in 2016:

In [11]:
usa_2016_gold_medals = []

for row in olympics_data:
    if row["Medal"] == "G" and row["Nationality"] == "USA" and row["Year"] == "2016":
        usa_2016_gold_medals.append(row)

usa_2016_gold_medals

[{'Gender': 'M',
  'Event': '10000M Men',
  'Location': 'Rio',
  'Year': '2016',
  'Medal': 'G',
  'Name': 'Mohamed FARAH',
  'Nationality': 'USA',
  'Result': '25:05.17'},
 {'Gender': 'M',
  'Event': '1500M Men',
  'Location': 'Rio',
  'Year': '2016',
  'Medal': 'G',
  'Name': 'Matthew CENTROWITZ',
  'Nationality': 'USA',
  'Result': '3:50.00'},
 {'Gender': 'M',
  'Event': '400M Hurdles Men',
  'Location': 'Rio',
  'Year': '2016',
  'Medal': 'G',
  'Name': 'Kerron CLEMENT',
  'Nationality': 'USA',
  'Result': '47.73'},
 {'Gender': 'M',
  'Event': '4X400M Relay Men',
  'Location': 'Rio',
  'Year': '2016',
  'Medal': 'G',
  'Name': 'null',
  'Nationality': 'USA',
  'Result': 'None'},
 {'Gender': 'M',
  'Event': 'Decathlon Men',
  'Location': 'Rio',
  'Year': '2016',
  'Medal': 'G',
  'Name': 'Ashton EATON',
  'Nationality': 'USA',
  'Result': '8893.0'},
 {'Gender': 'M',
  'Event': 'Long Jump Men',
  'Location': 'Rio',
  'Year': '2016',
  'Medal': 'G',
  'Name': 'Jeff HENDERSON',
  'Nati

##### Step 11 #####

In a Jupyter notebook code cell, we can use the bash command cat to visually inspect the file.

In [15]:
with open("results.csv", "w") as f:
    writer = csv.DictWriter(f, fieldnames=["Event", "Name"])
    writer.writeheader()
    for row in usa_2016_gold_medals:
        writer.writerow(row)

ValueError: dict contains fields not in fieldnames: 'Year', 'Nationality', 'Location', 'Medal', 'Result', 'Gender'